# ex3 — Force vs. Magnet Position

**Original problem**: The source exercise modeled a magnet moving through a coil and computing the induced current — a Transient Eddy Current problem.

**Constraint 1 — no Transient solver**: The Student license does not support the Transient solver (`Ansys Electronics Desktop Student does not support Maxwell Transient solution`). This was worked around by treating the magnet's position as a static design variable (`magnet_z0`) and running a parametric sweep across 17 discrete positions (-5 cm to 3 cm, 0.5 cm step) using the Magnetostatic solver, rather than a continuous time-domain simulation.

**Constraint 2 — custom field expressions can't be re-evaluated across a sweep**: The natural next step was computing the flux linkage through the coil via a custom surface-integral expression in the Field Calculator. The expression itself registered successfully and was visible in `available_report_quantities()`. However, evaluating that custom expression across parametric variations consistently failed under the Student license (`Operation error: 'ClcEval'`), regardless of whether Save Fields was enabled or the geometry was set as Model vs. Non-Model, and regardless of whether a single value or the full sweep was requested. This points to a specific Student-license limitation: custom Field Calculator expressions (surface/volume integrals) cannot be re-evaluated across parametric sweep variations, even though the same expression evaluates correctly for a single nominal solve.

**Final approach**: The problem was reframed around the force acting on the magnet — a standard quantity computed and cached directly at solve time via `assign_force()`, which requires no post-hoc re-evaluation and therefore remains stable under the Student license across the full parametric sweep.

In [ ]:
import pandas as pd
import numpy as np
import os
import ansys.aedt.core
import math
import shutil
import time
import matplotlib.pyplot as plt


In [ ]:
## Initialize AEDT Desktop session and create Maxwell 3D project/design ##

DT = ansys.aedt.core.Desktop(version="2025.2", non_graphical=False, student_version=True)
DT.disable_autosave()

sol_type = "Magnetostatic"

M3D = ansys.aedt.core.maxwell.Maxwell3d(solution_type=sol_type)
oDesign = M3D.odesign


In [ ]:
## Set up output directory for results ##

proj_name = "Week1_ex3"
desi_name = "Week1_ex3"

proj_dir = os.getcwd() + f"\\{proj_name}"
print(proj_dir)

os.makedirs(proj_dir, exist_ok=True)

proj_path = f"{proj_dir}\\{proj_name}.aedt"


In [ ]:
## Save project and apply design name ##

proj = M3D.oproject
proj.SaveAs(proj_path, True)

M3D.rename_design(desi_name, save=False)


In [ ]:
## Create geometry ##

M3D.modeler.model_units = "cm"

# Register magnet position as a design variable
M3D["magnet_z0"] = "-1cm"

# Create the magnet
center = [0, 0, "magnet_z0"]
origin = [0, 1.3, "magnet_z0"]
height = 2
magnet = M3D.modeler.create_polyhedron(orientation=None, center=center, origin=origin, height=height,
                                        num_sides=36, name="Magnet", material=None)

# Create coil terminal
origin = [0, 1.5, -2.5]
sizes = [1, 5]
coil_terminal = M3D.modeler.create_rectangle(orientation="YZ", origin=origin, sizes=sizes,
                                              name="Coil_Terminal", material=None, is_covered=True)

# Create coil by sweeping the terminal around the axis
coil = coil_terminal.clone()
M3D.modeler.sweep_around_axis(assignment=coil, axis="Z", sweep_angle=360, draft_angle=0, number_of_segments=0)
coil.name = "Coil"

# Create simulation region
region = M3D.modeler.create_region(pad_value=50, pad_type='Percentage Offset', name='Region')


In [ ]:
## Assign materials ##

magnet_material = M3D.materials.duplicate_material(material="NdFe35", name="Magnet_Material", properties=None)
M3D.assign_material(assignment=magnet, material=magnet_material.name)
magnet_material.set_magnetic_coercivity(value=-890000, x=0, y=0, z=1)

M3D.assign_material(assignment=coil, material="copper")


In [ ]:
## Assign force calculation (force acting on the magnet) ##

# Force is computed and cached at solve time, so it must be assigned here, before running the analysis;
# adding it afterward would require re-solving.
force = M3D.assign_force(assignment="Magnet", is_virtual=True, force_name="Force_on_Magnet")


In [ ]:
## Create and configure analysis setup ##

my_setup = M3D.create_setup(name="Setup1")
my_setup.props["MaximumPasses"] = 6
my_setup.props["PercentError"] = 1
my_setup.update()


In [ ]:
## Assign mesh operations ##

M3D.mesh.assign_length_mesh(assignment=coil, inside_selection=True, maximum_length=1,
                             maximum_elements=700, name="coil_mesh")
M3D.mesh.assign_length_mesh(assignment=magnet, inside_selection=True, maximum_length=1,
                             maximum_elements=400, name="magnet_mesh")


In [ ]:
## Set up and run parametric sweep (with Save Fields enabled) ##

sweep = M3D.parametrics.add(variable="magnet_z0", start_point="-5cm", end_point="3cm",
                             step="0.5cm", variation_type="LinearStep", name="PosSweep")

sweep.props["ProdOptiSetupDataV2"]["SaveFields"] = True
sweep.update()

sweep.analyze()


In [ ]:
## (Check) List available force-related quantities ##

# Confirms the exact quantity name (e.g. Force_on_Magnet.Force_z) before referencing it in get_solution_data() below
available = M3D.post.available_report_quantities(report_category="Fields")
print([q for q in available if "force" in q.lower()])


In [ ]:
## Extract result data (force acting on the magnet) ##
## Match the expressions value to the exact name confirmed in the cell above ##

data = M3D.post.get_solution_data(
    expressions="Force_on_Magnet.Force_z",
    setup_sweep_name="Setup1 : LastAdaptive",
    primary_sweep_variable="magnet_z0",
    variations={"magnet_z0": ["All"]},
)

z_positions = np.array(data.primary_sweep_values, dtype=float)
force_z = np.array(data.data_real("Force_on_Magnet.Force_z"), dtype=float)

df = pd.DataFrame({"magnet_z0_cm": z_positions, "Force_z_N": force_z})
df = df.sort_values("magnet_z0_cm").reset_index(drop=True)
df


In [ ]:
## Plot results ##

plt.figure(figsize=(7, 5))
plt.plot(df["magnet_z0_cm"], df["Force_z_N"], marker="o")
plt.xlabel("Magnet position magnet_z0 [cm]")
plt.ylabel("Force on Magnet, Z-direction [N]")
plt.title("Magnetic Force vs Magnet Position")
plt.grid(True)
plt.tight_layout()
plt.savefig("Week1_ex3_force_plot.png", dpi=150)
plt.show()

df.to_csv("Week1_ex3_force_result.csv", index=False)


In [ ]:
# Save final results

M3D.save_project()
